# Local GPU Training (warm-start)

Fine-tune from `rabah2026/wav2vec2-large-xlsr-53-arabic-quran-v_final` on a **local
NVIDIA CUDA GPU** (e.g. RTX 3050/3060), using the project's vocab + the
bootstrap→finetune staging trainer.

**Bahasa Indonesia:** Training warm-start di GPU lokal (CUDA). **Bukan untuk M1 Mac**
(wav2vec2-large di CPU/MPS terlalu lambat) — jalankan di mesin dengan NVIDIA GPU.

Pre-reqs on the target machine: NVIDIA driver + CUDA, `ffmpeg`, `uv`, and this repo
cloned locally. Run cells top-to-bottom from the repo root. Audio/dataset/checkpoints
live on the local disk; nothing touches Google Drive.

In [ ]:
# GPU check — CUDA must be available (this is a GPU notebook, not the M1)
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No CUDA GPU found — this notebook is for a local NVIDIA GPU machine, not the M1 Mac.'

In [ ]:
# Optional env tweaks
import os
os.environ.setdefault('HF_HOME', os.path.expanduser('~/.cache/huggingface'))
# os.environ['HF_TOKEN'] = 'hf_xxx'   # uncomment ONLY to push the trained model to the Hub

## 1. Environment setup

Create the venv and install deps with `uv` (the project's tool). The notebook kernel
should be the project's `.venv` — run `uv sync`, then select the `.venv` kernel
(`.venv/bin/python`) in Jupyter if prompts appear.

In [ ]:
!uv sync
!uv run python scripts/cloud_preflight.py   # GPU / ffmpeg / disk checks

## 2. Data + training

`configs/local_warmstart.yaml` points at:
- **base**: `rabah2026/...-quran-v_final` (warm-start; head re-init to the project vocab)
- **data**: everyayah audio (Husary) + quran.com text → local disk
- **trainer**: `train_manual.py` with `auto_stage` (bootstrap head-only → finetune last encoder layer)

Re-run any cell to resume (download/build/vocab cached; training resumes from `latest`).
For FROM-SCRATCH training instead of warm-start, switch the `--config` below to
`configs/local_3050.yaml`.

In [ ]:
# Download everyayah audio + quran.com text (resumable) → local data/raw
!uv run python scripts/download.py --config configs/local_warmstart.yaml

In [ ]:
# Build the HF dataset (join audio + quran.com text, by_surah split)
!uv run python scripts/build.py --config configs/local_warmstart.yaml

In [ ]:
# Build the diacritics-aware CTC vocab from the quran.com corpus
!uv run python scripts/build_vocab.py --config configs/local_warmstart.yaml

In [ ]:
# Warm-start training — LONG-RUNNING. Re-run to resume after an interrupt.
# IMPORTANT: use train_manual.py (staging + head re-init), NOT scripts/train.py.
!uv run python scripts/train_manual.py --config configs/local_warmstart.yaml

## 3. After training

Final + best checkpoints are at `data/artifacts/checkpoints/local_warmstart/`
(`final/`, `best/`). Copy `final/` to your backend to serve it, or run the inference
sanity check below.

In [ ]:
# Sanity: run the corrector on Al-Fatiha 1:1 with the trained final checkpoint
FINAL = 'data/artifacts/checkpoints/local_warmstart/final'
AUDIO = 'data/raw/audio/Husary_128kbps_Mujawwad/001001.wav'
!uv run python scripts/correct.py --model-dir {FINAL} --audio {AUDIO} \
    --text 'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ'